# Exponential Smoothing

## Overview

Exponential smoothing methods weight recent observations more heavily than older ones, with weights decaying exponentially into the past. They are fast, robust, and competitive with ARIMA on many real series.

**Three levels of complexity:**

| Model | Handles | Parameters |
|---|---|---|
| Simple Exponential Smoothing (SES) | Level only (no trend/season) | α |
| Holt's Linear | Level + trend | α, β |
| Holt-Winters | Level + trend + seasonality | α, β, γ |

All parameters (α, β, γ) are estimated by minimising one-step-ahead forecast error. Range [0,1]: values near 1 weight recent observations heavily; values near 0 give more weight to history.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_error

rng = np.random.default_rng(42)
dates = pd.date_range("2012-01-01", periods=144, freq="MS")
trend    = np.linspace(2.5, 4.2, 144)
seasonal = 0.8*np.sin(2*np.pi*np.arange(144)/12)
noise    = rng.normal(0, 0.25, 144)
ts = pd.Series(trend+seasonal+noise, index=dates, name="nitrate")
train, test = ts.iloc[:120], ts.iloc[120:]
print(f"Train: {len(train)}, Test: {len(test)}")

---
## Simple Exponential Smoothing

In [ ]:
# SES: no trend or seasonal component -- appropriate for level-only series
ses = SimpleExpSmoothing(train, initialization_method="estimated").fit(optimized=True)
print(f"SES: alpha={ses.params['smoothing_level']:.4f}")
ses_fc = ses.forecast(len(test))
rmse_ses = np.sqrt(mean_squared_error(test, ses_fc))
print(f"SES Test RMSE: {rmse_ses:.4f}")
print("Note: SES will miss trend and seasonality -- shown for baseline comparison")

---
## Holt-Winters (Additive and Multiplicative)

In [ ]:
# Additive: seasonal fluctuations constant in size
hw_add = ExponentialSmoothing(train, trend="add", seasonal="add",
                              seasonal_periods=12,
                              initialization_method="estimated").fit(optimized=True)
# Multiplicative: seasonal fluctuations proportional to level
hw_mul = ExponentialSmoothing(train, trend="add", seasonal="mul",
                              seasonal_periods=12,
                              initialization_method="estimated").fit(optimized=True)
for model, name in [(hw_add,"Additive"),(hw_mul,"Multiplicative")]:
    p = model.params
    print(f"\nHolt-Winters {name}:")
    print(f"  alpha={p['smoothing_level']:.3f}, beta={p['smoothing_trend']:.3f}, gamma={p['smoothing_seasonal']:.3f}")
    fc = model.forecast(len(test))
    print(f"  Test RMSE={np.sqrt(mean_squared_error(test,fc)):.4f}, MAE={mean_absolute_error(test,fc):.4f}")

---
## Forecast Plot with Confidence Intervals

In [ ]:
hw_fc = hw_add.forecast(len(test))
# Simulate prediction intervals via bootstrap residuals
resid = hw_add.resid
boot_forecasts = []
rng2 = np.random.default_rng(0)
for _ in range(500):
    boot_resid = rng2.choice(resid.dropna(), size=len(test), replace=True)
    boot_forecasts.append(hw_fc.values + boot_resid)
boot_forecasts = np.array(boot_forecasts)
pi_lo = np.percentile(boot_forecasts, 2.5, axis=0)
pi_hi = np.percentile(boot_forecasts, 97.5, axis=0)
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(train, color="steelblue", lw=1.5, label="Train")
ax.plot(test,  color="navy",      lw=1.5, label="Test (actual)")
ax.plot(test.index, hw_fc, color="#e74c3c", lw=2, linestyle="--", label="HW Forecast")
ax.fill_between(test.index, pi_lo, pi_hi, alpha=0.2, color="#e74c3c", label="95% Bootstrap PI")
ax.set_ylabel("Nitrate (mg/L)"); ax.set_title("Holt-Winters Additive Forecast")
ax.legend(); plt.tight_layout(); plt.show()

---
## Model Comparison

In [ ]:
models = {
    "SES":            ses.forecast(len(test)),
    "HW Additive":    hw_add.forecast(len(test)),
    "HW Multiplicative": hw_mul.forecast(len(test)),
}
rows = []
for name, fc in models.items():
    rows.append({
        "Model": name,
        "RMSE":  np.sqrt(mean_squared_error(test, fc)),
        "MAE":   mean_absolute_error(test, fc),
        "MAPE":  np.mean(np.abs((test.values - fc.values)/test.values))*100
    })
print(pd.DataFrame(rows).set_index("Model").round(4))
print("\nHolt-Winters should outperform SES substantially when trend+season present")

---

## Common Pitfalls

**1. Using multiplicative seasonality when the series contains zeros or near-zeros**  
Multiplicative seasonality divides and multiplies by the seasonal index. If the series contains zeros, this produces undefined or extreme values. Use additive seasonality for data with zero or negative values.

**2. Choosing additive vs multiplicative seasonality by default rather than by inspection**  
Additive is appropriate when seasonal swing is roughly constant over time. Multiplicative is appropriate when the seasonal swing grows proportionally with the level. Plot the series and examine whether seasonal amplitude increases — do not guess.

**3. Interpreting smoothing parameters without checking if they hit boundaries**  
If alpha, beta, or gamma are estimated at exactly 0 or 1, the optimisation has hit a boundary and the model may be misspecified. A gamma near 0 means seasonal patterns are barely updated — check that the seasonal period is correctly specified.

**4. Not comparing to a naive benchmark**  
A seasonal naive forecast (last year's same month) is a strong benchmark for seasonal data. If Holt-Winters does not substantially outperform seasonal naive, the model is not adding value. Always include naive benchmarks.

**5. Using Holt-Winters for series with changing seasonal patterns**  
Holt-Winters assumes stable seasonal patterns. If the seasonal cycle is shifting (e.g. climate-driven phenological change), the seasonal component will lag behind reality. Use dynamic or structural time series models for evolving seasonality.
---
*python_methods_library - Samantha McGarrigle*